In [ ]:
"""

Evaluating IPSL-CM7 simulations

"""

In [1]:
import xarray as xr
import basal_melt_NEMO.metrics_functions as mf
import basal_melt_NEMO.useful_functions as uf
from basal_melt_NEMO.constants import *
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [2]:
sns.set_context('paper')

In [3]:
%matplotlib qt5

QStandardPaths: error creating runtime directory '/run/user/2784' (Permission denied)


In [4]:
inputpath = '/thredds/tgcc/store/burgardc/FORMATTED/'
mask_path = '/data/cburgard/TOOLS/'
inputpath_interim='/data/cburgard/PREPARE_FORCING/PREPARE_PRESCRIBED_MELT/interim/'
inputpath_raw2 = '/data/cburgard/PREPARE_FORCING/PREPARE_CAVITY_MASKS/raw/'

outputpath = '/data/cburgard/EVAL_IPSLCM/'

In [5]:
rrun = 'opencav-presc04'

In [ ]:
### FILE T PREPARATION

In [ ]:
file_T_list = []

#for yy in range(5,9):
#    file_T = xr.open_dataset(inputpath + 'opencav-presc02_18'+str(yy)+'00101_18'+str(yy)+'91231_1M_grid_T_varofint.nc')
#    file_T_list.append(file_T)

#file_T_all = xr.concat(file_T_list, dim='time_counter').rename({'time_counter':'time'})

file_T_all = xr.open_dataset(inputpath + 'opencav-presc02_18500101_18891231_1M_grid_T_varofint.nc').rename({'time_counter':'time'})

In [ ]:
### FILE U PREPARATION

In [ ]:
file_U_list = []

#for yy in range(5,9):
#    file_U = xr.open_dataset(inputpath + 'opencav-presc02_18'+str(yy)+'00101_18'+str(yy)+'91231_1M_grid_U_varofint.nc')
#    file_U_list.append(file_U)

#file_U_all = xr.concat(file_U_list, dim='time_counter').rename({'time_counter':'time'})
file_U_all = xr.open_dataset(inputpath + 'opencav-presc02_18500101_18891231_1M_grid_U_varofint.nc').rename({'time_counter':'time'})

In [ ]:
### FILE V PREPARATION

In [ ]:
file_V_list = []

#for yy in range(5,9):
#    file_V = xr.open_dataset(inputpath + 'opencav-presc02_18'+str(yy)+'00101_18'+str(yy)+'91231_1M_grid_V_varofint.nc')
#    file_V_list.append(file_V)
#
#file_V_all = xr.concat(file_V_list, dim='time_counter').rename({'time_counter':'time'})
file_V_all = xr.open_dataset(inputpath + 'opencav-presc02_18500101_18891231_1M_grid_V_varofint.nc').rename({'time_counter':'time'})

In [ ]:
### FILE TRC

In [6]:
file_bgc = xr.open_dataset(inputpath + 'opencav-presc04_18500101_18591231_1M_ptrc_T.nc')

In [14]:
file_bgc['DIC'].isel(time_counter=0).max('deptht').plot()

In [ ]:
mask_land_3D = (file_T_all['so'].isel(time=0) > 0).rename('ls_mask').drop('time').load()

In [ ]:
mask_land_3D.isel(deptht=45).plot()

In [ ]:
cellarea = file_T_all['cell_area'].where(mask_land_3D > 0)


In [ ]:
u_vertsum = (file_U_all['uocetr_eff'] * (mask_land_3D.rename({'deptht':'depthu'}) > 0)).sum('depthu')

In [ ]:
ocean_masks = xr.open_dataset(mask_path + 'basin_masks_orca1_nemo4p2.nc')
isf_masks = xr.open_dataset(inputpath_interim + 'masks_for_eORCA1_prescribedmeltinopencav.nc')

In [ ]:
domain_cfg = xr.open_dataset(inputpath_raw2 + 'eORCA1.4.3_OpenSeas_OpenAllCav_ModStraights/eORCA1.4.3_OpenSeas_OpenAllCav_ModStraights_domain_cfg.nc')

In [ ]:
cellvolume = domain_cfg['e1t'] * domain_cfg['e2t'] * domain_cfg['e3t_0']
cellvolume = cellvolume.rename({'z': 'deptht'})

In [ ]:
mask_open_ocean = mask_land_3D.isel(deptht=0).drop('deptht')
contshelf = mask_open_ocean & (domain_cfg['bathy_metry'] < 1500)
mask_all = mask_land_3D.sum('deptht') > 0

Define bottom and ocean-ice interface

In [ ]:
# pour tous les points
vert_diff_plus_all = (mask_land_3D - mask_land_3D.shift(deptht=1)).isel(deptht=range(1,len(mask_land_3D.deptht)))
vert_diff_minus_all = (mask_land_3D - mask_land_3D.shift(deptht=-1)).isel(deptht=range(1,len(mask_land_3D.deptht)))

In [ ]:
# dernier point d'océan avant la glace - somme de toutes les profondeurs où la différence est positive (donc la transition glace-océan)
ice_depth_all = (mask_land_3D.deptht * vert_diff_plus_all).where(vert_diff_plus_all > 0).sum('deptht').astype('float')
# dernier point d'océan avant le fond - somme de toutes les profondeurs où la différence est positive (donc la transition océan-fond)
bot_depth_all = (mask_land_3D.deptht * vert_diff_minus_all).where(vert_diff_minus_all > 0).sum('deptht').astype('float')

In [ ]:
ice_depth_all = ice_depth_all.where(ice_depth_all > 0,0)
bot_depth_all = bot_depth_all.where(bot_depth_all !=0, mask_land_3D.deptht.values[0])

In [ ]:
bottom_idx = (mask_land_3D.deptht <= domain_cfg['bathy_metry']).sum('deptht')
ice_idx = (mask_land_3D.deptht <= domain_cfg['isf_draft']).sum('deptht')

Define the open cavities

In [ ]:
mask_open = isf_masks['mask_isf_open']

In [ ]:
ID_open_list = [21,66,117,124,127,128] #67,125,

Cavity quantities

In [ ]:
def compute_cavity_quantities(file_T_all,ID_open_list,mask_land_3D,mask_open,cellarea,cellvolume,bot_depth_all):

    ds = xr.Dataset()
    
    #Tmean cav
    #Smean cav

    print('cavmean')
    mean_T_cav_list = []
    mean_S_cav_list = []
    
    for kisf in ID_open_list:
        if kisf == 66:
            msk_domain = (mask_land_3D) & ((mask_open == 66) | (mask_open == 67))
        elif kisf == 124:
            msk_domain = (mask_land_3D) & ((mask_open == 124) | (mask_open == 125))
        else:
            msk_domain = (mask_land_3D) & (mask_open == kisf)
            
        mean_T_cav = uf.weighted_mean(file_T_all['thetao'].where(msk_domain),['deptht','x','y'], cellvolume.where(msk_domain))
        mean_S_cav = uf.weighted_mean(file_T_all['so'].where(msk_domain),['deptht','x','y'], cellvolume.where(msk_domain))
        mean_T_cav_list.append(mean_T_cav.assign_coords({'ID': kisf}))
        mean_S_cav_list.append(mean_S_cav.assign_coords({'ID': kisf}))
    
    ds['Tcav_mean'] = xr.concat(mean_T_cav_list, dim='ID')
    ds['Scav_mean'] = xr.concat(mean_S_cav_list, dim='ID')

    print('cavbot')
    #Tbot cav
    #Sbot cav
    bottom_cell = bot_depth_all
    
    mean_T_bot_list = []
    mean_S_bot_list = []
    
    for kisf in ID_open_list:
        if kisf == 66:
            msk_isf = (mask_open == 66) | (mask_open == 67)
        elif kisf == 124:
            msk_isf = (mask_open == 124) | (mask_open == 125)
        else:
            msk_isf = mask_open == kisf
            
        mean_T_bot = uf.weighted_mean(file_T_all['thetao'].sel(deptht=bottom_cell).where(msk_isf),['deptht','x','y'],cellarea.where(msk_isf))
        mean_S_bot = uf.weighted_mean(file_T_all['so'].sel(deptht=bottom_cell).where(msk_isf),['deptht','x','y'],cellarea.where(msk_isf))
        mean_T_bot_list.append(mean_T_bot.assign_coords({'ID': kisf}))
        mean_S_bot_list.append(mean_S_bot.assign_coords({'ID': kisf}))
    ds['Tbot_mean'] = xr.concat(mean_T_bot_list, dim='ID')
    ds['Sbot_mean'] = xr.concat(mean_S_bot_list, dim='ID')


    # Melt in the open cavities
    print('cavmelt')
    mean_melt_list = []
    for kisf in ID_open_list:
        if kisf == 66:
            msk_isf = (mask_open == 66) | (mask_open == 67)
        elif kisf == 124:
            msk_isf = (mask_open == 124) | (mask_open == 125)
        else:
            msk_isf = mask_open == kisf
            
        #mean_melt = uf.weighted_mean(file_T_all['iceshelf'].where(msk_isf),['deptht','x','y'],cellarea.where(msk_isf))
        mean_melt_Gt_yr = (file_T_all['iceshelf'].where(msk_isf) * yearinsec * cellarea.where(msk_isf) * 10**(-12)).sum(['x','y','deptht'])
        mean_melt_list.append(mean_melt_Gt_yr.assign_coords({'ID': kisf}))
    ds['melt_Gt_yr'] = xr.concat(mean_melt_list, dim='ID')

    return ds

In [ ]:
def compute_biogeochem_inside_quantities(file_BGC_all, bot_depth_all,cellarea,mask_open_ocean,contshelf):

    ds = xr.Dataset()
    
    #Tmean cav
    #Smean cav
    
    mean_vv_list = []

    for vv in file_BGC_all.vars:
        for kisf in ID_open_list:
            mean_vv_cav = uf.weighted_mean(file_T_all[vv].where((mask_land_3D) & (mask_open == kisf)),['deptht','x','y'], cellvolume)
            mean_vv_cav_list.append(mean_T_cav.assign_coords({'ID': kisf}))
    
        ds[vv+'cav_mean'] = xr.concat(mean_vv_cav_list, dim='ID')
    
        #Tbot cav
        #Sbot cav
        bottom_cell = bot_depth_all
        
        mean_vv_bot_list = []
        
        for kisf in ID_open_list:
            mean_vv_bot = uf.weighted_mean(file_T_all[vv].sel(deptht=bottom_cell).where(mask_open == kisf),['x','y'],cellarea)
            mean_vv_bot_list.append(mean_vv_bot.assign_coords({'ID': kisf}))
        ds[vv+'bot_mean'] = xr.concat(mean_vv_bot_list, dim='ID')


    return ds

Non-cavity quantities

In [ ]:
def bottom_prop_reg(var,lon,lat,reg,bottom_cell,contshelf):

    # mask regions for bottom properties
    mask_regions = xr.Dataset()
    mask_regions['AMU'] = (lon >= -109.64) & (lon <= -102.23) & (lat >= -75.80) & (lat <= -71.66) & (contshelf)
    mask_regions['WROSS'] = (lon >= 157.100) & (lon <= 173.333) & (lat >= -78.130) & (lat <= -74.040) & (contshelf)
    mask_regions['EROSS'] = (lon >= -176.790) & (lon <= -157.820) & (lat >= -78.870) & (lat <= -77.520) & (contshelf)
    mask_regions['WWED'] = (lon >= -65.130) & (lon <= -53.020) & (lat >= -75.950) & (lat <= -72.340) & (contshelf)
    mask_regions['EWED'] = (lon >= -45.647) & (lon <= -32.253) & (lat >= -78.632) & (lat <= -76.899) & (contshelf)
    mask_regions['PRYDZ'] = (lon >= 65.) & (lon <= 80.) & (lat >= -75.) & (lat <= -65.) & (contshelf)
    
    return var.sel(deptht=bottom_cell).where(mask_regions[reg])

In [ ]:
def compute_outside_quantities(file_T_all, u_vertsum, bot_depth_all,cellarea,contshelf):

    bottom_cell = bot_depth_all

    ds = xr.Dataset()
    
    #Tbot in the main seas
    #Sbot in the main seas
    
    for rreg in ['AMU','WROSS','EROSS','WWED','EWED','PRYDZ']:
    
        Treg = bottom_prop_reg(file_T_all['thetao'],file_T_all.nav_lon,file_T_all.nav_lat,rreg,bottom_cell,contshelf)
        Sreg = bottom_prop_reg(file_T_all['so'],file_T_all.nav_lon,file_T_all.nav_lat,rreg,bottom_cell,contshelf)
        
        ds['Tbot_'+rreg] = uf.weighted_mean(Treg,['deptht','x','y'],cellarea.where(np.isfinite(Treg))) #.sel(x=Treg.x,y=Treg.y)
        ds['Sbot_'+rreg] = uf.weighted_mean(Sreg,['deptht','x','y'],cellarea.where(np.isfinite(Sreg))) #.sel(x=Sreg.x,y=Sreg.y)

    # ACC
    ds['ACC'] = u_vertsum.sel(x=220,y=range(79,107)).sum('y')
    ds['ACC'] = ds['ACC']/10**6

    #Weddell Gyre
    uocetr_Wed = u_vertsum.where(uf.in_range(cellarea.nav_lat,[-66.50,-60.40]) & uf.in_range(cellarea.nav_lon,[-31.25,37.50]), drop=True)
    ds['wed_gyre'] = uocetr_Wed.cumsum('y').max(['y','x'])/10**6

    # Ross Gyre
    uocetr_Ross = u_vertsum.where(uf.in_range(cellarea.nav_lat,[-72.650,-61.600]) & ((cellarea.nav_lon <= -135.75) | (cellarea.nav_lon >= 360-168.500)), drop=True)   
    ds['ross_gyre'] = uocetr_Ross.cumsum('y').max(['y','x'])/10**6

    return ds

In [ ]:
def compute_seaice_quantities(file_SI_all,cellarea):

    ds = xr.Dataset()

    
    lon = cellarea.lon
    lat = cellarea.lat
    
    mask_Arc = (lat >= 50) 
    mask_Ant = (lat <= -50) 
        
    file_ice_15 = file_SI_all['siconc'] > 0.15

    # SI extent

    sie_Ant = (file_ice_15.where(mask_Ant) * cellarea).sum(['x','y']).rename({'time_counter':'time'}).load()
    sie_Arc = (file_ice_15.where(mask_Arc) * cellarea).sum(['x','y']).rename({'time_counter':'time'}).load()

    print('Computing sea-ice extent')

    vvar = sie_Arc.where(sie_Arc['time.month'] == 3, drop=True).squeeze()/10**12
    ds['mar_sie_arc'] = vvar.assign_coords({'time': vvar['time.year']})
    
    vvar = sie_Arc.where(sie_Arc['time.month'] == 9, drop=True).squeeze()/10**12
    ds['sep_sie_arc'] = vvar.assign_coords({'time': vvar['time.year']})

    vvar = sie_Ant.where(sie_Ant['time.month'] == 2, drop=True).squeeze()/10**12
    ds['feb_sie_ant'] = vvar.assign_coords({'time': vvar['time.year']})

    vvar = sie_Ant.where(sie_Ant['time.month'] == 9, drop=True).squeeze()/10**12
    ds['sep_sie_ant'] = vvar.assign_coords({'time': vvar['time.year']})

    # SI area

    sia_Ant = (file_seaice_n['siconc'].where(mask_Ant) * cellarea).sum(['x','y']).rename({'time_counter':'time'}).load()
    sia_Arc = (file_seaice_n['siconc'].where(mask_Arc) * cellarea).sum(['x','y']).rename({'time_counter':'time'}).load()

    print('Computing sea-ice area')
    
    vvar = sia_Arc.where(sia_Arc['time.month'] == 3, drop=True).squeeze()/10**12
    ds['mar_sia_arc'] = vvar.assign_coords({'time': vvar['time.year']})

    vvar = sia_Arc.where(sia_Arc['time.month'] == 9, drop=True).squeeze()/10**12
    ds['sep_sia_arc'] = vvar.assign_coords({'time': vvar['time.year']})

    vvar = sia_Ant.where(sia_Ant['time.month'] == 2, drop=True).squeeze()/10**12
    ds['feb_sia_ant'] = vvar.assign_coords({'time': vvar['time.year']})

    vvar = sia_Ant.where(sia_Ant['time.month'] == 9, drop=True).squeeze()/10**12
    ds['sep_sia_ant'] = vvar.assign_coords({'time': vvar['time.year']})
    
    # SI volume

    siv_Ant = (file_seaice_n['sivolu'].where(mask_Ant) * cellarea).sum(['x','y']).rename({'time_counter':'time'}).load()
    siv_Arc = (file_seaice_n['sivolu'].where(mask_Arc) * cellarea).sum(['x','y']).rename({'time_counter':'time'}).load()

    print('Computing sea-ice volume')
    
    vvar = siv_Arc.where(siv_Arc['time.month'] == 3, drop=True).squeeze()/10**12
    ds['mar_siv_arc'] = vvar.assign_coords({'time': vvar['time.year']})

    vvar = siv_Arc.where(siv_Arc['time.month'] == 9, drop=True).squeeze()/10**12
    ds['sep_siv_arc'] = vvar.assign_coords({'time': vvar['time.year']})

    vvar = siv_Ant.where(siv_Ant['time.month'] == 2, drop=True).squeeze()/10**12
    ds['feb_siv_ant'] = vvar.assign_coords({'time': vvar['time.year']})
        
    vvar = siv_Ant.where(siv_Ant['time.month'] == 9, drop=True).squeeze()/10**12
    ds['sep_siv_ant'] = vvar.assign_coords({'time': vvar['time.year']}) 

    return ds

In [ ]:
kisf = 66
del ds_cavity

In [ ]:
# For runs including cavities
#ds_cavity = compute_cavity_quantities(file_T_all,ID_open_list,mask_land_3D,mask_open,cellarea,cellvolume,bot_depth_all)
ds_cavity.to_netcdf(outputpath + rrun + '_metrics_open_cavities.nc')

In [ ]:
for vv in ['Tcav_mean','Scav_mean','Tbot_mean','Sbot_mean','melt_Gt_yr']:
    plt.figure()
    ds_cavity[vv].sel(ID=124).plot()
    plt.title(vv)

In [ ]:
file_T_all['thetao'].sel(deptht=bot_depth_all)

In [ ]:
ds_cavity['melt_Gt_yr'].sum(['x','y']).sel(ID=124).plot()


In [ ]:
# For runs not including cavities
#ds_noncavity = compute_outside_quantities(file_T_all, u_vertsum, bot_depth_all,cellarea,contshelf)
ds_noncavity.to_netcdf(outputpath + rrun + 'metrics_outside_cavities.nc')

In [ ]:
for vv in ds_noncavity.variables:
    plt.figure()
    plt.title(vv)
    ds_noncavity[vv].plot()

In [ ]:
ds_seaice = compute_seaice_quantities(file_SI_all,cellarea)
ds_seaice.to_netcdf(outputpath + rrun + 'metrics_seaice.nc')

In [ ]:
mask_open_ocean

In [ ]:
contshelf

This is outside of the cavity

In [ ]:
#Tbot in the main seas
#Sbot in the main seas

mean_T_bot_list = []
mean_S_bot_list = []

for kisf in ID_open_list:
    mean_T_bot = file_T_all['thetao'].isel(deptht=bottom_cell).where(mask_open == kisf).mean(['x','y'])
    mean_S_bot = file_T_all['so'].isel(deptht=bottom_cell).where(mask_open == kisf).mean(['x','y'])
    mean_T_bot_list.append(mean_T_bot.assign_coords({'ID': kisf}))
    mean_S_bot_list.append(mean_S_bot.assign_coords({'ID': kisf}))
mean_T_bot_all = xr.concat(mean_T_bot_list, dim='ID')
mean_S_bot_all = xr.concat(mean_S_bot_list, dim='ID')
for rreg in ['AMU','WROSS','EROSS','WWED','EWED','PRYDZ']:

    Treg = bottom_prop_reg(file_T_all['thetao'],lon,lat,reg,bottom_cell)
    Sreg = bottom_prop_reg(file_T_all['so'],lon,lat,reg,bottom_cell)
    
    ds['Tbot_'+rreg] = uf.weighted_mean(Treg.where(mask_open_ocean & contshelf),['x','y'],cellarea)
    ds['Sbot_'+rreg] = uf.weighted_mean(Sreg.where(mask_open_ocean & contshelf),['x','y'],cellarea)

    #plt.figure()
    #mean_T_cav.plot()
    #plt.title('Mean T in cavity '+str(kisf))
    #sns.despine()

    #plt.figure()
    #mean_S_cav.plot()
    #plt.title('Mean S in cavity '+str(kisf))
    #sns.despine()

In [ ]:
kisf = 21

f = plt.figure()
f.set_size_inches(8.25/1.5, 8.25)

ax={}

i = 0

for i,vv in enumerate(['Tmean_cav','Tmean_bot','Tfront_bot','Smean_cav','Smean_bot','Sfront_bot']):
    ax[i] = f.add_subplot(2,3,i+1)
    ax[i].plot(ds[vv].sel(ID=kisf))
    ax[i].set_title(vv+' '+str(kisf))
sns.despine()
    #ax[i].axhline(y=var_obs_mean.sel(var=vv), color='black', linewidth=2)
    #ax[i].fill_between(x=np.arange(0,100),y1=var_obs_mean.sel(var=vv)-var_obs_std.sel(var=vv), y2=var_obs_mean.sel(var=vv)+var_obs_std.sel(var=vv), color='grey',alpha=0.2)


In [ ]:
Tbot cav
Sbot cav
Tbot front
Sbot front
fwf flux (integrated for the open ice shelves)